# Data Preparation

The point of this notebook is to process the raw data from [investing.com](https://www.investing.com/) and [stooq.pl](https://stooq.pl/) into a format which is easy to use in the rest of the project. 

## Setup

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
# Find project root (folder that contains .git)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

# Set working directory to root
os.chdir(ROOT)

print("Now working in:", Path.cwd())

Now working in: C:\Users\couch\OneDrive\Assignments\Master Thesis\Repo


## Stock Data

For stock data, it must be merged into one DataFrame of log-returns (`results/processed_data/stock_data.parquet`) while also setting the values to missing if the day in question can't be used for the analysis. Here are the guidelines for that:

- LTS can no longer be part of the porfolio after 2022-06-02. It's reasonable for the investor to not consider this company as a worthwhile part of a portfolio if the company it's merging with is also a part of the possible investments. It's reasonable that the investor would turn off the ability to buy this due to [the merger being announced](https://pap-mediaroom.pl/biznes-i-finanse/grupa-lotos-sa-102022-uzgodnienie-planu-polaczenia-grupy-lotos-sa-z-pkn-orlen-sa).
- PGN can no longer be part of the portfolio after 2022-07-28. This case is pretty much the same as the one above. [The merger was announced](https://pap-mediaroom.pl/biznes-i-finanse/pkn-orlen-sa-372022-uzgodnienie-planu-polaczenia-pomiedzy-pkn-orlen-sa-pgnig-sa) so a more conservative investor would take it as a cue to back out of the company and only consider the parent from then on.
- PLY can no longer be part of the portfolio after 2020-09-20. It's expected that the investor managing their portfolio would back out of it being part of later porfolios after the company [was officially sold to Iliad SA](https://www.parkiet.com/komunikaty-espi/art27211681-play-communications-s-a).

In [3]:
def load_and_merge_csvs(folder_path):
    folder = Path(folder_path)
    series_list = []

    for file_path in folder.glob("*.csv"):
        col_name = file_path.stem.upper()

        # `thousands=','` parses comma-separated string numbers into floats automatically
        df = pd.read_csv(file_path, parse_dates=["Date"], index_col="Date", thousands=",")

        # Safely convert to numeric in case string/whitespace values persist
        price_series = pd.to_numeric(df["Price"].astype(str).str.replace(",", ""), errors="coerce").rename(col_name)

        series_list.append(price_series)

    merged_df = pd.concat(series_list, axis=1, join="outer").sort_index()

    return merged_df

In [4]:
# Map each column to its strict cutoff date
cutoffs = {"LTS": "2022-06-03", "PGN": "2022-07-29", "PLY": "2020-09-21"}

# Merge the data into one DataFrame
stock_df = load_and_merge_csvs("data/stocks")

# Filter out dates after the cutoffs
for col, cutoff_date in cutoffs.items():
    if col in stock_df.columns:
        stock_df.loc[stock_df.index > cutoff_date, col] = np.nan

# Calculate log-returns
stock_df = np.log(stock_df).diff()[1:]

display(stock_df)

,ALR,CDR,CPS,DNP,EBP,JSW,KGH,LPP,LTS,MBK,MDV,OPL,PEO,PGE,PGN,PKN,PKO,PLY,PZU,TPE
Date,,,,,,,,,,,,,,,,,,,,
2014-01-03,-0.009629,-0.006342,-0.001011,NaN,-0.010414,0.009294,-0.003378,0.011834,-0.007074,-0.017298,-0.004692,0.005099,-0.010880,0.001238,-0.021053,-0.014708,-0.015544,NaN,-0.016329,-0.013668
2014-01-07,-0.019666,-0.017503,-0.023013,NaN,-0.034886,-0.023588,-0.021377,-0.030703,-0.030714,-0.028988,-0.017684,-0.008172,-0.029895,-0.009326,-0.047534,-0.009927,-0.015267,NaN,-0.019091,-0.025553
2014-01-08,0.009443,-0.005311,-0.026207,NaN,0.018792,0.002082,0.001296,-0.006255,-0.009915,0.024385,-0.019777,-0.005141,0.031297,-0.007524,-0.008147,-0.002378,0.001303,NaN,0.006554,0.007034
2014-01-09,0.000000,-0.003557,-0.005325,NaN,0.000000,-0.051194,-0.032013,-0.030700,-0.001608,-0.034839,0.011035,0.000000,-0.031297,0.008772,0.020244,-0.014389,-0.010734,NaN,-0.024862,-0.016490
2014-01-10,0.000000,-0.002378,0.014312,NaN,-0.008011,-0.017259,-0.005362,0.000000,-0.002902,0.018718,-0.016822,0.008214,0.022858,-0.018257,-0.028457,0.019139,0.008388,NaN,-0.016298,-0.004762
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-23,-0.021100,-0.029177,-0.038268,-0.047370,-0.007881,-0.003083,-0.040557,0.000986,NaN,-0.038603,-0.042069,-0.012320,-0.013223,0.038466,NaN,0.025867,-0.012052,NaN,-0.019634,-0.003781
2026-07-24,-0.001872,-0.000442,-0.000619,-0.002776,0.006673,-0.019880,0.004955,-0.007913,NaN,-0.003828,-0.017242,0.002751,0.003429,-0.011385,NaN,0.000898,-0.005987,NaN,0.000000,-0.008878
2026-07-27,0.011552,0.036036,0.029908,0.040516,0.016192,0.017175,-0.003300,0.022583,NaN,0.015605,0.034191,0.004796,0.016550,-0.022191,NaN,-0.032576,0.013235,NaN,0.013982,-0.012821


In [5]:
# Identify the start and end of non-NaN data for each column
valid_started = stock_df.notna().cummax()
valid_ended = stock_df.notna()[::-1].cummax()[::-1]

# Create a mask for rows strictly between the first and last valid values
middle_mask = valid_started & valid_ended

# Find where NaNs exist inside that active range
internal_nans = stock_df.isna() & middle_mask

# Identify columns containing internal NaNs
cols_with_internal_nans = internal_nans.any()[internal_nans.any()].index.tolist()

if not cols_with_internal_nans:
    print("All NaNs exist strictly on the edges (no gaps in the middle)")
else:
    print(f"Columns with internal gaps: {cols_with_internal_nans}")

All NaNs exist strictly on the edges (no gaps in the middle)


In [6]:
display(stock_df.isna().sum())

ALR       0
CDR       0
CPS       0
DNP     826
EBP       0
JSW       0
KGH       0
LPP       0
LTS    1037
MBK       0
MDV       0
OPL       0
PEO       0
PGE       0
PGN     998
PKN       0
PKO       0
PLY    2357
PZU       0
TPE       0
dtype: int64

In [7]:
# Save the DataFrame to a single Parquet file
stock_df.to_parquet("results/processed_data/stock_data.parquet", engine="pyarrow", compression="snappy")

# Bond Yield Data

The risk-free rate of return is derived from the yield of 10-year treasury bond fron the Polish Government. The data used here comes from [stooq.pl](https://stooq.pl/). Since the returns saved on this site have a derivative nature, the minimum trading price for each month is taken as the conservative estimate of the risk-free rate of return. The results are saved as a JSON which maps the year and month (in the format `YYYY-MM`) to a value (`results/processed_data/risk_free_rate.json`).

In [8]:
bond_df = pd.read_csv("data/10yply_b_m.csv")

# Standardize date keys to 'YYYY-MM'
bond_df["Data"] = pd.to_datetime(bond_df["Data"]).dt.strftime("%Y-%m")

# Clean numeric daily return values (handles commas); 252 is the expected number of yearly trading days
bond_df["Najnizszy"] = (
    1 + pd.to_numeric(bond_df["Najnizszy"].astype(str).str.replace(",", "."), errors="coerce") / 100
) ** (1 / 252) - 1

# Export key-value pairs to JSON
bond_dict = dict(zip(bond_df["Data"], bond_df["Najnizszy"], strict=True))

with open("results/processed_data/risk_free_rate.json", "w", encoding="utf-8") as f:
    json.dump(bond_dict, f, indent=2)

# Index Returns

Finally, it's worthwhile to safe the returns of the index itself in a file (`results/processed_data/index_returns.parquet`). These are simple returns and not log-returns.

In [9]:
def process_simple_returns(csv_file_path):
    # Read CSV with Date as datetime index and thousands separator handling
    df = pd.read_csv(csv_file_path, parse_dates=["Date"], index_col="Date", thousands=",")

    # Clean Price column and convert to float
    prices = pd.to_numeric(df["Price"].astype(str).str.replace(",", "."), errors="coerce")

    # Calculate simple returns
    simple_returns = prices.pct_change().rename("Return")[1:]

    # Convert to DataFrame
    returns_df = simple_returns.to_frame()

    return returns_df

In [10]:
index_df = process_simple_returns("data/wig20.csv")
display(index_df)

,Return
Date,
2014-01-03,-0.010491
2014-01-07,-0.023125
2014-01-08,0.005865
2014-01-09,-0.016011
2014-01-10,0.001011
...,...
2026-08-11,0.006116
2026-08-12,0.004066
2026-08-13,-0.014423


In [11]:
index_df.isna().sum()

Return    0
dtype: int64

In [12]:
index_df.to_parquet("results/processed_data/index_returns.parquet", engine="pyarrow", compression="snappy")